# MLOps — Training Data from Feature Store

Loads the preprocessed MovieLens 1M splits from the Feature Store (Delta Lake on MinIO via DuckDB).

**Prerequisites**
- MinIO and Redis containers are running: `docker compose up -d`
- Delta tables exist in MinIO: run `preprocess.py` first
- Feast registry is set up: run `python featurestorage/trainning/apply.py --registry-only`

## 1. Preprocessing

Reads raw `.dat` files from MinIO `raw-data` bucket, runs feature engineering,
and writes Delta Lake tables to `s3://processed/{train,val,test}`.

In [4]:
import sys, os
sys.path.insert(0, os.path.abspath("."))

from preprocess import MovieLensPreprocessor

preprocessor = MovieLensPreprocessor()
train_data_raw, val_data_raw, test_data_raw, metadata = preprocessor.process_all()

print("\n[Metadata summary]")
for k, v in metadata.items():
    if not isinstance(v, dict):
        print(f"  {k}: {v}")

Loading MovieLens 1M from MinIO bucket 'raw-data'...
  Users: 6040, Movies: 3883, Ratings: 1000209

[Feature engineering]
  Age groups: {1: 0, 18: 1, 25: 2, 35: 3, 45: 4, 50: 5, 56: 6}
  Gender encoding: {'F': 0, 'M': 1}
  Genre vocabulary size: 19 (including padding)
  Time of Day distribution:
    0 Matinee    : 318906
    1 Prime Time : 243539
    2 Late Night : 437764

[Sequences]
Building user sequences...
  Built sequences for 6038 users
  Total unique items: 3628

[Split]
Splitting sequences...
  Train: 818363, Val: 6038, Test: 6038

[Saving to Delta Lake in MinIO]
  [train] Deleted existing data from Delta table
  [train] Written 818,363 new rows → s3://processed/train
  [val] Deleted existing data from Delta table
  [val] Written   6,038 new rows → s3://processed/val
  [test] Deleted existing data from Delta table
  [test] Written   6,038 new rows → s3://processed/test
  Saved metadata                  → s3://processed/metadata.json

[Metadata summary]
  num_items: 3629
  num_

## 2. Load datasets from Feature Store

In [5]:
from pathlib import Path
from featurestorage.trainning import TrainingFeatureStore

# feature_store.yaml lives in featurestorage/trainning/, not the project root
REPO_PATH = str(Path("featurestorage/trainning").resolve())

with TrainingFeatureStore(repo_path=REPO_PATH) as store:
    train_data = store.get_dataset_records("train")
    val_data   = store.get_dataset_records("val")
    test_data  = store.get_dataset_records("test")

print(f"train : {len(train_data):>7,} records")
print(f"val   : {len(val_data):>7,} records")
print(f"test  : {len(test_data):>7,} records")

train : 818,363 records
val   :   6,038 records
test  :   6,038 records


## 3. Inspect the data

In [6]:
import pprint

print("=== train_data[0] structure ===")
pprint.pprint(train_data[0])

print("\n=== key types ===")
r = train_data[0]
for k, v in r.items():
    if isinstance(v, dict):
        print(f"  {k}: dict  → {v}")
    elif isinstance(v, list):
        inner = type(v[0]).__name__ if v else "?"
        print(f"  {k}: list[{inner}] len={len(v)}")
    else:
        print(f"  {k}: {type(v).__name__} = {v}")

=== train_data[0] structure ===
{'genre_seq': [[8, 0, 0]],
 'item_seq': [2914],
 'target': 1153,
 'target_time': 2,
 'time_seq': [2],
 'user_id': 1,
 'user_profile': {'age_idx': 0, 'gender_idx': 0, 'occupation': 10}}

=== key types ===
  user_id: int = 1
  item_seq: list[int] len=1
  genre_seq: list[list] len=1
  time_seq: list[int] len=1
  target: int = 1153
  target_time: int = 2
  user_profile: dict  → {'age_idx': 0, 'gender_idx': 0, 'occupation': 10}


In [7]:
import pickle

# Load expected reference
ref_path = "/Volumes/ktran2/MSE/DeepLearning/Projects/MSA-VH-GRP/Mamba4Rec/data/processed/"
with open(ref_path + "train_data.pkl", "rb") as f:
    ref_train = pickle.load(f)
with open(ref_path + "val_data.pkl", "rb") as f:
    ref_val = pickle.load(f)
with open(ref_path + "test_data.pkl", "rb") as f:
    ref_test = pickle.load(f)

# Compare counts
assert len(train_data) == len(ref_train), f"train mismatch: {len(train_data)} vs {len(ref_train)}"
assert len(val_data)   == len(ref_val),   f"val mismatch"
assert len(test_data)  == len(ref_test),  f"test mismatch"

# Compare keys
assert set(train_data[0].keys()) == set(ref_train[0].keys()), \
    f"key mismatch: {set(train_data[0].keys())} vs {set(ref_train[0].keys())}"

# Compare first record value-by-value
r, ref = train_data[0], ref_train[0]
assert r["user_id"]     == ref["user_id"]
assert r["item_seq"]    == ref["item_seq"]
assert r["genre_seq"]   == ref["genre_seq"]
assert r["time_seq"]    == ref["time_seq"]
assert r["target"]      == ref["target"]
assert r["target_time"] == ref["target_time"]
assert r["user_profile"] == ref["user_profile"]

print("All assertions passed.")
print(f"  train : {len(train_data):>7,}  ==  {len(ref_train):>7,} (reference)")
print(f"  val   : {len(val_data):>7,}  ==  {len(ref_val):>7,} (reference)")
print(f"  test  : {len(test_data):>7,}  ==  {len(ref_test):>7,} (reference)")
print(f"  keys  : {sorted(train_data[0].keys())}")

All assertions passed.
  train : 818,363  ==  818,363 (reference)
  val   :   6,038  ==    6,038 (reference)
  test  :   6,038  ==    6,038 (reference)
  keys  : ['genre_seq', 'item_seq', 'target', 'target_time', 'time_seq', 'user_id', 'user_profile']
